# Does the marked verb describe vertical movement?

Run each cell from the top. You will change a reading rule, train a classifier, and read a disagreement. The four provisional labels come from Qwen3.5-9B: movement happens (CT+), is negated (CT−, written CT-), is left open (Uu), or the verb has another sense (NM). Record your own predictions and readings before viewing the results. You may ask an assistant to explain Python errors.

### Load the class data
Run the setup cell once. It pairs each passage with its label and saved embedding by ID. The two cells marked **Edit** are where you change the experiment.

In [ ]:
#@title Load passages, labels, and vectors
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/movement-classification.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = pd.read_json(bundle.open("class7/passages.jsonl"), lines=True).set_index("id")
    labels = pd.read_json(bundle.open("class7/llm-labels.jsonl"), lines=True).set_index("id")
    with np.load(bundle.open("class7/embeddings.npz")) as saved:
        vectors = pd.DataFrame(saved["vectors"].astype("float32"), index=saved["ids"])

passages["reference"] = labels["label"]

train = passages[passages["split"] == "train"]
dev = passages[passages["split"] == "dev"].dropna(subset=["reference"])
test = passages[passages["split"] == "test"].dropna(subset=["reference"])
train_vectors = vectors.loc[train.index]
train_labels = train["reference"]

def show_passage(row):
    print(row["author"], row["title"], row["sentence"])
    print(row["previous"])
    print(row["left"] + "【" + row["target"] + "】" + row["right"])
    print(row["next"])

### 1. Read before predicting
Read the marked verb and the neighboring sentences. What falls? Choose a label before continuing.

In [ ]:
passage = dev.loc["dev-0054"]
show_passage(passage)

**Edit 1.** This rule gives NM when the words immediately before 落ちる end with the cue, and CT+ otherwise. Predict its answer for the passage above. In Python, `left.endswith(cue)` tests whether the left context ends with that cue. Run the next two cells, then replace `眠りに` (“into sleep”) with a phrase from the passage that means “the voice [of a skylark]” and ends in が. Rerun both cells.

In [ ]:
cue = "眠りに"

In [ ]:
left = passage["left"]
if left.endswith(cue):
    prediction = "NM"
else:
    prediction = "CT+"
print('Rule', prediction)
print('Qwen', passage["reference"])

Try the same rule on 本国の土佐へ【落ち】て行った. Write down its prediction and your own label. Read the surrounding sentences below. Which words tell you whether someone moved vertically?

In [ ]:
show_passage(dev.loc["dev-0046"])

Now reveal Qwen’s reading. If yours differs, explain which words support the difference.

In [ ]:
print("Qwen", dev.loc["dev-0046", "reference"])

### 2. Train a classifier
From here on, the input is the saved embedding for each passage. Each of the 420 training passages has a vector of numbers and a provisional label. `fit` uses those pairs to learn a classifier. You will use the fitted model to predict labels for passages from other works.

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(train_vectors, train_labels)
print(len(train_labels), 'training passages')

Compare the predictions with Qwen’s readings on the validation set. Find the Uu row. How many Uu passages did the classifier recover, out of how many? What can you conclude about CT- recall from its row? Rows are Qwen labels and columns are predictions.

In [ ]:
dev_predictions = model.predict(vectors.loc[dev.index])
matrix = pd.crosstab(dev["reference"], dev_predictions, colnames=["prediction"])
labels_in_order = ["CT+", "CT-", "Uu", "NM"]
matrix.reindex(index=labels_in_order, columns=labels_in_order, fill_value=0)

### 3. Read a disagreement on the test set
Run the next cell to count agreements with Qwen and collect the disagreements.

In [ ]:
test_predictions = model.predict(vectors.loc[test.index])
results = test.assign(prediction=test_predictions)
agrees = results["prediction"] == results["reference"]
print(agrees.sum(), "/", len(results))
disagreements = results[~agrees]

**Edit 2.** Start with case 0. Run the two cells below and read the marked verb in context. Record your label and one phrase that supports it before running the comparison.

In [ ]:
case_number = 0

In [ ]:
case = disagreements.iloc[case_number]
show_passage(case)

Compare your reading with the two labels. Which reading can you support from the text? Change `case_number` to 1, rerun the selection and passage cells, and make a fresh judgment before revealing those labels.

In [ ]:
print("Qwen", case["reference"])
print('Classifier', case["prediction"])